# MICE stands for Multivariate Imputation By Chained Equations

First use mean to impute and then use linear model to predict value by removing value of linear model predicted to again null of one column and use its neighbours columns data as X input and its own value of other rows as output and then predict the value by passing the null value corresponding columns value as input and find value 
Then diffence the mean imputed value and predicted value if the difference is all zero then its predicted value is accurate otherwise we need to iterate it more until the difference becomes zero.
It can takes many iterations like 20,40,100 also.

In [7]:
import pandas as pd
import numpy as np 
from sklearn.linear_model import LinearRegression

In [4]:
df= (pd.read_csv('/Users/prajwalsubedi/Desktop/Data science/Data Sets/50_Startups.csv')[['R&D Spend','Administration','Marketing Spend','Profit']]/10000)

In [5]:
df.head()

,R&D Spend,Administration,Marketing Spend,Profit
0,16.534920,13.689780,47.178410,19.226183
1,16.259770,15.137759,44.389853,19.179206
2,15.344151,10.114555,40.793454,19.105039
3,14.437241,11.867185,38.319962,18.290199
4,14.210734,9.139177,36.616842,16.618794


In [8]:
np.random.seed(9)
df = df.sample(5)
df

,R&D Spend,Administration,Marketing Spend,Profit
21,7.838947,15.377343,29.973729,11.131302
37,4.406995,5.128314,19.702942,8.994914
2,15.344151,10.114555,40.793454,19.105039
14,11.994324,15.654742,25.651292,13.260265
44,2.217774,15.480614,2.833472,6.520033


In [9]:
df = df.iloc[:,0:-1]
df

,R&D Spend,Administration,Marketing Spend
21,7.838947,15.377343,29.973729
37,4.406995,5.128314,19.702942
2,15.344151,10.114555,40.793454
14,11.994324,15.654742,25.651292
44,2.217774,15.480614,2.833472


In [11]:
df.iloc[1,0] = np.nan
df.iloc[3,1] = np.nan
df.iloc[-1,-1] = np.nan

/var/folders/nl/75k6nf4573d4dnj713sbnl5w0000gn/T/ipykernel_56585/766688332.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[1,0] = np.nan
/var/folders/nl/75k6nf4573d4dnj713sbnl5w0000gn/T/ipykernel_56585/766688332.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[3,1] = np.nan
/var/folders/nl/75k6nf4573d4dnj713sbnl5w0000gn/T/ipykernel_56585/766688332.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[-1,-1]

In [12]:
df.head()

,R&D Spend,Administration,Marketing Spend
21,7.838947,15.377343,29.973729
37,NaN,5.128314,19.702942
2,15.344151,10.114555,40.793454
14,11.994324,NaN,25.651292
44,2.217774,15.480614,NaN


In [13]:
# Step 1 - Impute all missing values with mean of respective col

df0 = pd.DataFrame()

df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean())
df0['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())

In [14]:
# 0th Iteration
df0

,R&D Spend,Administration,Marketing Spend
21,7.838947,15.377343,29.973729
37,9.348799,5.128314,19.702942
2,15.344151,10.114555,40.793454
14,11.994324,11.525207,25.651292
44,2.217774,15.480614,29.030354


In [16]:
# Remove the col1 imputed value
df1 = df0.copy()

df1.iloc[1,0] = np.nan

df1

,R&D Spend,Administration,Marketing Spend
21,7.838947,15.377343,29.973729
37,NaN,5.128314,19.702942
2,15.344151,10.114555,40.793454
14,11.994324,11.525207,25.651292
44,2.217774,15.480614,29.030354


In [17]:
# Use first 3 rows to build a model and use the last for prediction

X = df1.iloc[[0,2,3,4],1:3]
X

,Administration,Marketing Spend
21,15.377343,29.973729
2,10.114555,40.793454
14,11.525207,25.651292
44,15.480614,29.030354


In [18]:
	
y = df1.iloc[[0,2,3,4],0]
y

21     7.838947
2     15.344151
14    11.994324
44     2.217774
Name: R&D Spend, dtype: float64

In [19]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[1,1:].values.reshape(1,2))

/opt/anaconda3/envs/DS/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([23.02178743])

In [20]:
df1.iloc[1,0] = 23.14

In [21]:
df1

,R&D Spend,Administration,Marketing Spend
21,7.838947,15.377343,29.973729
37,23.140000,5.128314,19.702942
2,15.344151,10.114555,40.793454
14,11.994324,11.525207,25.651292
44,2.217774,15.480614,29.030354


In [23]:
# Remove the col2 imputed value

df1.iloc[3,1] = np.nan

df1

,R&D Spend,Administration,Marketing Spend
21,7.838947,15.377343,29.973729
37,23.140000,5.128314,19.702942
2,15.344151,10.114555,40.793454
14,11.994324,NaN,25.651292
44,2.217774,15.480614,29.030354


In [24]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[[0,1,2,4],[0,2]]
X

,R&D Spend,Marketing Spend
21,7.838947,29.973729
37,23.140000,19.702942
2,15.344151,40.793454
44,2.217774,29.030354


In [25]:
y = df1.iloc[[0,1,2,4],1]
y

21    15.377343
37     5.128314
2     10.114555
44    15.480614
Name: Administration, dtype: float64

In [26]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[3,[0,2]].values.reshape(1,2))

/opt/anaconda3/envs/DS/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([11.3823476])

In [27]:
df1.iloc[3,1] = 11.06


In [28]:
df1

,R&D Spend,Administration,Marketing Spend
21,7.838947,15.377343,29.973729
37,23.140000,5.128314,19.702942
2,15.344151,10.114555,40.793454
14,11.994324,11.060000,25.651292
44,2.217774,15.480614,29.030354


In [30]:
# Remove the col3 imputed value
df1.iloc[4,-1] = np.nan

df1

,R&D Spend,Administration,Marketing Spend
21,7.838947,15.377343,29.973729
37,23.140000,5.128314,19.702942
2,15.344151,10.114555,40.793454
14,11.994324,11.060000,25.651292
44,2.217774,15.480614,NaN


In [31]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[0:4,0:2]
X

,R&D Spend,Administration
21,7.838947,15.377343
37,23.140000,5.128314
2,15.344151,10.114555
14,11.994324,11.060000


In [32]:
y = df1.iloc[0:4,-1]
y

21    29.973729
37    19.702942
2     40.793454
14    25.651292
Name: Marketing Spend, dtype: float64

In [33]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[4,0:2].values.reshape(1,2))

/opt/anaconda3/envs/DS/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([27.33065522])

In [34]:
df1.iloc[4,-1] = 31.56

In [35]:
# After 1st Iteration
df1

,R&D Spend,Administration,Marketing Spend
21,7.838947,15.377343,29.973729
37,23.140000,5.128314,19.702942
2,15.344151,10.114555,40.793454
14,11.994324,11.060000,25.651292
44,2.217774,15.480614,31.560000


In [36]:
# Subtract 0th iteration from 1st iteration

df1 - df0

,R&D Spend,Administration,Marketing Spend
21,0.000000,0.000000,0.000000
37,13.791201,0.000000,0.000000
2,0.000000,0.000000,0.000000
14,0.000000,-0.465207,0.000000
44,0.000000,0.000000,2.529646


#### Go further iterations until the difference become all zero